# Frozen-policy TV monitoring

Evaluate exact terminal-policy probabilities on Hypergrid/Regular Tree and
conditional probabilities on a saved RNA support. Compare empirical `M_TV`,
probabilistic `B_TV`, and flow-consistency scores across checkpoints.

**Scope:** RNA quantities concern the selected subgraph. Generic comparison plots
aggregate training seeds; they do **not** reproduce final Figure 10's fixed-policy
MC shading or terminal-cache union. See [protocol details](../../README.md#final-paper-metrics-and-scope).

The formulas, exact probabilities, sampling, and plots are in
[analysis_utils.py](analysis_utils.py), in the numbered TV sections.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "code/train.py").is_file() and (p / "requirements.txt").is_file())
for folder in (ROOT / "code", ROOT / "code/notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
from analysis_utils import AnalysisSession, TVAnalysis

## 1. Select runs, checkpoints, and sampling budget

`None` preserves environment overrides. Defaults: 100 trajectories per direction,
one MC repeat, and ten evenly spaced checkpoints. Repeats use seeds 42 onward.
For a small development check, select a single run and a few existing checkpoints.
Use five repeats for the Figure 4 sampling protocol.

In [ ]:
# None keeps the GF_* environment setting, or its documented default.
PROFILE = None       # e.g. "smoke", "main", "ablation", "grid-long", "rna-long"
RUN_ROOT = None      # e.g. "outputs/quickstart"; a suite or single run
FIGURE_ROOT = None   # e.g. "figures/my-analysis"; inside this repository
STRICT = None        # True rejects incomplete/incompatible analyses

session = AnalysisSession.from_environment(
    "tv_monitoring.ipynb", root=ROOT, profile=PROFILE, run_root=RUN_ROOT,
    figure_root=FIGURE_ROOT, strict=STRICT, warn_h16_target=True,
)

In [ ]:
SAMPLES = None      # GF_TV_SAMPLES; default 100
REPEATS = None      # GF_TV_REPEATS; default 1
CHECKPOINTS = None  # GF_TV_CHECKPOINTS, or e.g. [500, 1000]
DEVICE = None       # GF_DEVICE; defaults to CUDA if available

analysis = TVAnalysis(
    session, samples=SAMPLES, repeats=REPEATS,
    checkpoints=CHECKPOINTS, device=DEVICE,
)

## 2. Evaluate each saved policy

`M_TV` uses backward/reference trajectories; `B_TV` uses both forward and backward
samples. The confidence term is `2 log(2/alpha)/n_forward + 2 log(2/alpha)/n_backward`.
The separately labeled legacy bound is diagnostic only. Estimates do not replace
the assumptions required for a probabilistic certificate.

In [ ]:
raw = analysis.evaluate()

## 3. Calibrate and compare monitoring signals

For complete ten-checkpoint series, fit the linear calibration on the first five
checkpoints and evaluate it on the last five. Constant or incomplete series remain
missing where the metric is undefined. Exact FCS and MC FCS are exported separately.

In [ ]:
summary = analysis.calibrate()

## 4. Plot generic method comparisons

MC repeats are averaged inside each training run before summarizing independent
training seeds. A single training seed therefore has no training-seed SD.
For RNA, `GF_RNA_SUPPORT` selects a shared cache; otherwise `rna-long` selects the
unique matching StableTeacher cache. This general analysis keeps at most 10,000
states and does not construct the final Figure 10 union.

In [ ]:
comparison = analysis.plot_comparisons()

## 5. Optional RNA support sweep

Disabled unless `GF_RNA_SWEEP=1`. Uses the 300k StableTeacher run and retained top-K
supports. A supplied `GF_RNA_ORACLE_SUPPORT` must contain the top 200,000 states
with exhaustive normalization metadata; see [oracle preparation](rna_oracle_preparation.ipynb).
Captured reward mass is normalized over all `4**14` states, not just the oracle.

In [ ]:
support_sweep = analysis.sweep_supports()

## 6. Save the audit and check completeness

Results, support hashes, sampling settings, and diagnostics are saved alongside
the plots. `GF_STRICT=1` rejects incomplete checkpoint evaluations.

In [ ]:
manifest = analysis.finish()